# MNIST Correlation Experiment

Experiment: Correlation between neuron activation and outgoing edge weights in SSONN.

**Setup:**
- Dataset: MNIST
- Edge selection: by variance (`edge_score_aggregation="variance"`)
- Pure SSONN

**Metrics tracked in Comet ML:**
- Pearson & Spearman correlations: binary/ReLU activation × selected/unselected/all edges
- Aggregate statistics (mean activation, mean edge weight, fraction selected)
- Scatter plots at expansion events
- Per-SubLinear layer correlations

In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
from functools import partial
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'

print(device)

################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
# from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [ ]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=28 * 28, hidden_size=16):
        super(SimpleFCN, self).__init__()
        self.fc_output = nn.Linear(input_size, 10)


    def forward(self, x):
        x = self.fc_output(x)
        return x

In [ ]:
hyperparams = {
    "num_epochs": 50,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()),
    "edge_score_aggregation": "variance",
    "expansion_thresholds": {"fc_output": 0.4},
    "pruning_thresholds": {"fc_output": 1},
    "plateau_threshold": 0.05,
    "min_epochs_between_expansions": 8,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 2000,
    "weight_decay": 0,
    "diff_threshold": 0.05,
    "t_max": 16,
    "warmup_epochs": 5
}

In [ ]:
# Dataset and Dataloader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

# Load dataset and split into train/validation sets
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=g)

# finder — fraction of train used for expansion metric (100% = full train)
finder_fraction = 1.0
n_finder = int(finder_fraction * len(train_dataset))
finder_dataset = Subset(train_dataset.dataset, train_dataset.indices[:n_finder])

batch_size = hyperparams['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g)
finder_loader = DataLoader(finder_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
torch.manual_seed(SEED)
model = SimpleFCN()
model = convert_to_scaled_net(model, layers_to_convert=[model.fc_output], device=device)

model.fc_output.dropout.p = 0.3

In [ ]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="mnist-correlation",
)
experiment.log_parameters(hyperparams)
experiment.set_name("mnist pure ssonn variance correlation")

In [ ]:
criterion = nn.CrossEntropyLoss()

def optimizer_factory(model_instance, current_lr=None, base_lr=None):
    if base_lr is None:
        base_lr = hyperparams['learning_rate']
    if current_lr is None:
        current_lr = base_lr
    
    new_optimizer = optim.SGD(
        model_instance.parameters(),
        lr=current_lr,
        weight_decay=hyperparams['weight_decay'],
        momentum=0.9
    )
    
    for group in new_optimizer.param_groups:
        group['initial_lr'] = base_lr 
        
    return new_optimizer

trained_model = train_ssonn(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    finder_loader=finder_loader,
    criterion=criterion,
    optimizer_factory=optimizer_factory,
    hyperparams=hyperparams,
    device=device,
    experiment=experiment,
    use_log_tool=True,
    log_correlations=True,
)

In [ ]:
experiment.end()

In [ ]:
torch.save(model, 'mnist_correlation.pt')